# 05 · deque, heapq, bisect

**How this notebook works**
- Run the setup cell below first. Then go top to bottom, running each cell with **Shift+Enter**.
- Gray **example** cells: run them, then change something and run again.
- **Try it** cells: your scratch space for each section.
- **Predict the output**: write your guess in the comment *before* running.
- **Exercises** at the bottom: write your code, run the cell, then run the `check(...)` cell under it.

In [ ]:
# ▶ Run this cell first (Shift+Enter). It loads the test helper and the same imports LeetCode gives you.
import sys, pathlib
sys.path.insert(0, str(next(p for p in (pathlib.Path.cwd() / "_primer", pathlib.Path.cwd() / "python-primer" / "_primer") if p.exists())))
from check import check, summary, lesson
lesson("ex05_deque_heap_bisect")

import bisect, heapq, math, random
from collections import Counter, OrderedDict, defaultdict, deque
from functools import cache, lru_cache
from itertools import accumulate, combinations, groupby, pairwise, permutations, product
from typing import List, Optional

**Warm-up (5 min):** before reading, close everything and write down every trap you remember from lessons 01–04. Then check them against your *My traps* notes in those notebooks.

Three tools that replace Java's `ArrayDeque`, `PriorityQueue`, and `Collections.binarySearch`.

## deque: a queue that's fast at both ends

In [ ]:
from collections import deque

q = deque([1, 2, 3])
q.append(4)          # right, O(1)
q.appendleft(0)      # left, O(1)
print(q)
print(q.pop())       # right, O(1)
print(q.popleft())   # left, O(1)   <- the reason to use it; list.pop(0) is O(n)
print(q[0], q[-1])   # peek both ends, O(1)

Indexing into the middle (`q[i]`) is O(n). Use a deque for queues, BFS, and sliding windows where things leave from the front.

**BFS template**, shortest distance in an unweighted graph:

In [ ]:
def bfs(graph, start):
    dist = {start: 0}
    q = deque([start])
    while q:
        node = q.popleft()
        for nxt in graph[node]:
            if nxt not in dist:          # mark when you ENQUEUE, not when you dequeue
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return dist

graph = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
print(bfs(graph, "a"))

**Building the graph.** Graph problems usually give an edge list, not an adjacency dict. Build it first:

In [ ]:
from collections import defaultdict
edges = [[0, 1], [1, 2], [3, 4]]
graph = defaultdict(list)
for a, b in edges:
    graph[a].append(b)
    graph[b].append(a)        # drop this line for directed edges
print(dict(graph))

Course Schedule style problems give `[course, prereq]` pairs, which are directed. Grid problems don't need a graph at all; the neighbors are the four directions (lesson 02).

**Iterative DFS:** same as BFS with a list as a stack (`stack.pop()` instead of `q.popleft()`). Use it when recursion might exceed the depth limit (lesson 06).

In [ ]:
graph = {0: [1], 1: [0, 2], 2: [1], 3: []}
start = 0
stack, seen = [start], {start}
while stack:
    node = stack.pop()
    for nxt in graph[node]:
        if nxt not in seen:
            seen.add(nxt)
            stack.append(nxt)
print(seen)                   # everything reachable from 0

Level by level (needed for many tree problems):

In [ ]:
tree = {1: [2, 3], 2: [4, 5], 3: [], 4: [], 5: []}    # node -> children
q = deque([1])
level = 0
while q:
    print("level", level, list(q))
    for _ in range(len(q)):              # len(q) is evaluated once, so this is exactly one level
        node = q.popleft()
        q.extend(tree[node])
    level += 1

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## heapq: a min-heap on a plain list

In [ ]:
import heapq

h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
print(h[0])                 # peek smallest, O(1)
print(heapq.heappop(h))     # smallest, O(log n)
nums = [9, 4, 7, 1]
heapq.heapify(nums)         # in place, O(n)
print(nums[0])
print(heapq.nlargest(2, [9, 4, 7, 1]))   # sorted list of the k largest

**Max-heap:** there isn't a simple flag. Push negatives and negate on the way out:

In [ ]:
h = []
for x in [5, 1, 8]:
    heapq.heappush(h, -x)
largest = -heapq.heappop(h)
print(largest)

**Tuples** compare element by element, so `(priority, item)` works. If two priorities tie, Python compares the items, and that crashes for things like `ListNode`. Add a tiebreaker:

In [ ]:
class Node:
    def __init__(self, val):
        self.val = val

a, b = Node(1), Node(1)
h = []
for i, node in enumerate([a, b]):
    heapq.heappush(h, (node.val, i, node))   # i is unique, so node is never compared
print(heapq.heappop(h)[1])

h2 = []
heapq.heappush(h2, (a.val, a))
try:
    heapq.heappush(h2, (b.val, b))           # equal priorities -> Python compares the nodes
except TypeError as e:
    print("without a tiebreaker:", e)

**Top-k pattern:** to keep the k *largest*, use a *min*-heap of size k and pop when it grows past k. The root is then the k-th largest. O(n log k).

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## bisect: binary search on a sorted list

In [ ]:
import bisect

a = [1, 3, 3, 3, 7]
print(bisect.bisect_left(a, 3))    # 1: first index with a[i] >= 3
print(bisect.bisect_right(a, 3))   # 4: first index with a[i] > 3
print(bisect.bisect_left(a, 4))    # 4: insertion point when absent
bisect.insort(a, 5)                # O(log n) search but O(n) insert
print(a)
print(bisect.bisect_right(a, 3) - bisect.bisect_left(a, 3))   # count of 3s

Count of `x` in a sorted list: `bisect_right(a, x) - bisect_left(a, x)`.

You still need to write binary search by hand, because the Binary Search section of Top 150 searches over answers and rotated arrays, not just sorted lists. The template that avoids off-by-one bugs:

In [ ]:
def first_at_least(a, target):
    lo, hi = 0, len(a)              # search space [lo, hi)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < target:         # condition that's False then True across the array
            lo = mid + 1
        else:
            hi = mid
    return lo                       # first index where the condition flips; equals bisect_left

a = [1, 3, 3, 3, 7]
print(first_at_least(a, 3), bisect.bisect_left(a, 3))
print(first_at_least(a, 4), first_at_least(a, 100))

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Predict the output

For each one: type your guess into the comment line, **then** run the cell. The output is the answer. If you were wrong, figure out why before moving on.

**1.** What does this print?

In [ ]:
# My guess: 
h = [5, 1, 4]
heapq.heapify(h)
print(h[0], heapq.heappop(h), h[0])

**2.** What does this print?

In [ ]:
# My guess: 
print(bisect.bisect_left([1, 2, 4], 3), bisect.bisect_right([1, 2, 2, 2], 2))

**3.** What does this print?

In [ ]:
# My guess: 
q = deque([1, 2, 3])
q.appendleft(0); q.pop()
print(list(q))

**4.** What does this print?

In [ ]:
# My guess: 
print(heapq.nlargest(2, [3, 9, 1, 9]))

<details><summary>Why (open after running all of them)</summary>

1. `1 1 4`
2. `2 4`
3. `[0, 1, 2]`
4. `[9, 9]`
</details>

---
# Exercises

Write your code in each cell (replace `raise NotImplementedError`), run it, then run the `check(...)` cell below it. `ok` means done; `FAIL` shows what was expected; `TODO` means not started.

### 1. `shortest_path_len`

graph maps node -> list of neighbors (unweighted). Return the number of edges on the shortest path from start to goal, or -1 if unreachable. BFS with a deque.

```
shortest_path_len(g, 'a', 'e')  ->  3
shortest_path_len(g, 'a', 'a')  ->  0
shortest_path_len(g, 'a', 'z')  ->  -1
```

In [ ]:
def shortest_path_len(graph: dict, start, goal) -> int:
    raise NotImplementedError

In [ ]:
check("shortest_path_len")

### 2. `level_sums`

levels_graph maps a node (an int value) to its list of children (a tree). Return the sum of node values at each depth, root level first. Use the `for _ in range(len(q))` level-by-level loop.

```
level_sums(tree, 1)  ->  [1, 5, 15]
```

In [ ]:
def level_sums(levels_graph: dict, root) -> list:
    raise NotImplementedError

In [ ]:
check("level_sums")

### 3. `kth_largest` · LeetCode 215

Solve it with a min-heap of size k. O(n log k). Don't sort.

```
kth_largest([3, 2, 1, 5, 6, 4], 2)           ->  5
kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4)  ->  4
```

In [ ]:
def kth_largest(nums: list, k: int) -> int:
    raise NotImplementedError

In [ ]:
check("kth_largest")

### 4. `merge_sorted_lists`

Merge k sorted lists into one sorted list using a heap of (value, list_index, element_index) tuples.

```
merge_sorted_lists([[], [0]])  ->  [0]
merge_sorted_lists([])         ->  []
```

In [ ]:
def merge_sorted_lists(lists: list) -> list:
    raise NotImplementedError

In [ ]:
check("merge_sorted_lists")

### 5. `last_stone_weight` · LeetCode 1046

Repeatedly smash the two heaviest stones; if they differ, the difference goes back. Return the last stone or 0. Max-heap via negation.

```
last_stone_weight([2, 7, 4, 1, 8, 1])  ->  1
last_stone_weight([1, 1])              ->  0
```

In [ ]:
def last_stone_weight(stones: list) -> int:
    raise NotImplementedError

In [ ]:
check("last_stone_weight")

### 6. `count_in_sorted`

How many times x appears in sorted list a, in O(log n), using bisect.

```
count_in_sorted([1, 2, 2, 2, 5], 2)  ->  3
count_in_sorted([1, 2, 2, 2, 5], 3)  ->  0
```

In [ ]:
def count_in_sorted(a: list, x) -> int:
    raise NotImplementedError

In [ ]:
check("count_in_sorted")

### 7. `search_insert` · LeetCode 35

Solve it, written by hand with the [lo, hi) template (no bisect).

```
search_insert([1, 3, 5, 6], 5)  ->  2
search_insert([1, 3, 5, 6], 2)  ->  1
search_insert([1, 3, 5, 6], 7)  ->  4
```

In [ ]:
def search_insert(nums: list, target: int) -> int:
    raise NotImplementedError

In [ ]:
check("search_insert")

### 8. `count_components`

Nodes are 0..n-1, edges are undirected pairs. Build an adjacency list with defaultdict(list), then count connected components using ITERATIVE DFS (a list as a stack). Must not hit the recursion limit on a 20,000-node chain.

```
count_components(5, [[0, 1], [1, 2], [3, 4]])                ->  2
count_components(3, [])                                      ->  3
count_components(20000, [[i, i + 1] for i in range(19999)])  ->  1
```

In [ ]:
def count_components(n: int, edges: list) -> int:
    raise NotImplementedError

In [ ]:
check("count_components")

### 9. `RecentCounter` (class) · LeetCode 933

Ping(t) records a request at time t (strictly increasing) and returns how many requests happened in [t - 3000, t]. Keep a deque and drop old ones from the left.

```
rc = RecentCounter()
rc.ping(1)     ->  1
rc.ping(100)   ->  2
rc.ping(3001)  ->  3
rc.ping(3002)  ->  3      (the ping at time 1 is now too old)
```

In [ ]:
class RecentCounter:

    def __init__(self):
        raise NotImplementedError

    def ping(self, t: int) -> int:
        raise NotImplementedError

In [ ]:
check("recent_counter")

---
## Wrap-up

In [ ]:
summary()

## My traps

*Double-click to edit.* Write the 3–5 things from this lesson that surprised you or that you got wrong, in your own words. You'll use these in the next warm-up and the exit test.

1. 
2. 
3. 

**Done?** All `ok` in the summary → commit and sync → tell Claude **"lesson 05 done."**